<a href="https://colab.research.google.com/github/aaminabenazir/capstone-project-part-4/blob/main/part4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests
import json
import re
from jsonschema import validate, ValidationError

# 1. Load API Key from your uploaded 'use.env' file
with open('use.env', 'r') as f:
    content = f.read().strip()
    api_key_value = content.split("=")[1] if "=" in content else content

os.environ['LLM_API_KEY'] = api_key_value
API_KEY = os.environ.get('LLM_API_KEY')

# 2. API Connection Function
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    # Use this specific URL:
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "My Project Part 4"
    }
    payload = {
        "model": "openai/gpt-3.5-turbo", # Changed model for troubleshooting 404 error
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    response = requests.post(url, headers=headers, json=payload, timeout=15)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        print(f"Error: {response.status_code}")
        return None

# 3. PII Guardrail
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9._+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

# 4. Pipeline for Track A
schema = {
    "type": "object",
    "properties": {
        "id": {"type": "number"},
        "name": {"type": "string"},
        "category": {"type": "string"},
        "is_active": {"type": "boolean"},
        "score": {"type": "number"}
    },
    "required": ["id", "name", "category", "is_active", "score"]
}

def process_pipeline(input_data):
    if has_pii(input_data):
        print("Input blocked: PII detected.")
        return None

    # Extract the 'raw' string from the input_data dictionary
    raw_text_to_extract = json.loads(input_data)['raw']

    system_prompt = (
        "You are a structured data extractor. Your task is to extract information from the following text "
        "and present it as a JSON object that strictly adheres to the provided JSON schema. "
        "Ensure all keys in the output JSON match the schema exactly (case-sensitive) and data types are correct.\n\n"
        "Mapping instructions:\n"
        "- 'ID' from the text should be mapped to the 'id' field in the JSON (as a number).\n"
        "- The name (e.g., 'John Doe') should be mapped to the 'name' field (as a string).\n"
        "- The department/category (e.g., 'Finance') should be mapped to the 'category' field (as a string).\n"
        "- The active status (e.g., 'True') should be mapped to the 'is_active' field (as a boolean).\n"
        "- The score (e.g., '95.5') should be mapped to the 'score' field (as a number).\n\n"
        "JSON Schema:\n" + json.dumps(schema, indent=2) + "\n\n"
        "Output only the JSON object, with no additional text or explanations."
    )

    user_prompt = f"Text to extract from: {raw_text_to_extract}"
    raw_response = call_llm(system_prompt, user_prompt, temperature=0)

    if not raw_response: return None

    try:
        data = json.loads(raw_response.strip())
        validate(instance=data, schema=schema)
        return data
    except (json.JSONDecodeError, ValidationError) as e:
        print(f"Validation Error: {e}")
        return None

# 5. Execution
if __name__ == "__main__":
    test_input = '{"raw": "ID 101, John Doe, Finance, True, 95.5"}'
    result = process_pipeline(test_input)
    print("Result:", result)